# ETL Pipeline – Slutuppgift Data Science

Denna notebook implementerar en ETL-pipeline som körs på två dataset:
- main dataset
- validation dataset


In [175]:
!pip install pandas==2.3.3 numpy python-dotenv matplotlib seaborn langchain langchain_groq

# IMPORTERA BIBLOTEK OCH LLM

Här kommer alla importer jag behöver för att kunna slutföra uppgiften.
Även här så kommer jag att ha min LLM.

In [176]:
import os
import time
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from dotenv import load_dotenv
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
assert GROQ_API_KEY, "Saknar GROQ_API_KEY i .env"


In [177]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

if llm:
    print("LLM is initialized")

LLM is initialized


# Ladda in och inspektera dataset


In [178]:
df_raw = pd.read_csv("nordtech_data.csv")

df_raw.head()

,order_id,orderrad_id,orderdatum,leveransdatum,produkt_sku,produktnamn,kategori,antal,pris_per_enhet,region,kundtyp,betalmetod,kund_id,leveransstatus,recension_text,recensionsdatum,betyg
0,ORD-2024-00001,ORD-2024-00001-1,2024-05-19,2024-05-22,SKU-WC001,Webbkamera HD,Tillbehör,1,SEK 799,Uppsala,Privat,Kort,KND-53648,Levererad,NaN,NaN,NaN
1,ORD-2024-00002,ORD-2024-00002-1,2024-12-02,5 december 2024,SKU-HB001,USB-C Hub 7-port,Tillbehör,1,549.00,Göteborg,Privat,Swish,KND-84095,Levererad,NaN,NaN,NaN
2,ORD-2024-00003,ORD-2024-00003-1,2024-12-31,2025-01-03,SKU-SD001,Extern SSD 1TB,Lagring,1,1199.00,NaN,Företag,Faktura,KND-91748,Levererad,Stämmer inte överens med produktbeskrivningen.,2025-01-12,2.0
3,ORD-2024-00003,ORD-2024-00003-2,2024-12-31,2025-01-03,SKU-SD002,Extern SSD 500GB,Lagring,10,699 kr,Stockholm,Företag,FAKTURA,KND-91748,Mottagen,"Leveransen tog lite längre än utlovat, men pro...",2025-01-14,3.0
4,ORD-2024-00003,ORD-2024-00003-3,2024-12-31,2025-01-03,SKU-MS001,Trådlös Mus X1,Tillbehör,1,399.00,Stockholm,Företag,Faktura,KND-91748,NaN,NaN,NaN,NaN


In [179]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2767 entries, 0 to 2766
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   order_id         2767 non-null   object 
 1   orderrad_id      2767 non-null   object 
 2   orderdatum       2767 non-null   object 
 3   leveransdatum    2767 non-null   object 
 4   produkt_sku      2767 non-null   object 
 5   produktnamn      2767 non-null   object 
 6   kategori         2767 non-null   object 
 7   antal            2767 non-null   object 
 8   pris_per_enhet   2767 non-null   object 
 9   region           2612 non-null   object 
 10  kundtyp          2767 non-null   object 
 11  betalmetod       2651 non-null   object 
 12  kund_id          2767 non-null   object 
 13  leveransstatus   2673 non-null   object 
 14  recension_text   1355 non-null   object 
 15  recensionsdatum  1355 non-null   object 
 16  betyg            1355 non-null   float64
dtypes: float64(1),

In [180]:
df_raw.tail()

,order_id,orderrad_id,orderdatum,leveransdatum,produkt_sku,produktnamn,kategori,antal,pris_per_enhet,region,kundtyp,betalmetod,kund_id,leveransstatus,recension_text,recensionsdatum,betyg
2762,ORD-2024-01654,ORD-2024-01654-3,2024-10-04,2024-10-07,SKU-LP003,Laptop Gaming X,Datorer,2,18999.00,stockholm,Företag,NaN,KND-99742,Levererad,"Förväntade mig mer för priset, men den duger.",2024-10-08,3.0
2763,ORD-2024-01655,ORD-2024-01655-1,2024-01-15,2024-01-17,SKU-MN003,"Bildskärm 32"" Curved",Bildskärmar,1,5999.00,Göteborg,Privat,Faktura,KND-83827,Returnerad,Stämmer inte överens med produktbeskrivningen.,2024-01-27,2.0
2764,ORD-2024-01656,ORD-2024-01656-1,2024-07-29,2024-07-31,SKU-MS002,Ergonomisk Mus Pro,Tillbehör,1,699.00,göteborg,Privat,faktura,KND-60471,Levererad,NaN,NaN,NaN
2765,ORD-2024-01656,ORD-2024-01656-2,2024-07-29,2024-07-31,SKU-MN001,"Bildskärm 27"" UHD",Bildskärmar,1,4999.00,Göteborg,Konsument,Faktura,KND-60471,Levererad,NaN,NaN,NaN
2766,ORD-2024-01657,ORD-2024-01657-1,2024-07-03,2024-07-05,SKU-MN002,"Bildskärm 24"" FHD",Bildskärmar,1,2499.00,STOCKHOLM,Privat,NaN,KND-26325,levererad,Överträffade mina förväntningar. 5 av 5!,2024-07-08,4.0


In [181]:
df_raw.sample(10)

,order_id,orderrad_id,orderdatum,leveransdatum,produkt_sku,produktnamn,kategori,antal,pris_per_enhet,region,kundtyp,betalmetod,kund_id,leveransstatus,recension_text,recensionsdatum,betyg
2696,ORD-2024-01613,ORD-2024-01613-2,2024-03-19,2024-03-22,SKU-SD002,Extern SSD 500GB,Lagring,1,699.00,Stockholm,Privat,Mastercard,KND-79417,Levererad,"Smidigt att beställa, produkten kom snabbt och...",2024-03-31,5.0
1559,ORD-2024-00945,ORD-2024-00945-2,2024-02-04,2024-02-06,SKU-MS001,Trådlös Mus X1,Tillbehör,en,"399,00",GBGB,Privat,Kort,KND-50350,Levererad,Fantastisk produkt! Fungerar precis som utlovat.,2024-02-09,4.0
2348,ORD-2024-01405,ORD-2024-01405-2,2024-05-31,2024-06-03,SKU-KB001,Mekaniskt Tangentbord K7,Tillbehör,5,1299.00,Stockholm,Företag,Faktura,KND-12673,Levererad,NaN,NaN,NaN
1328,ORD-2024-00808,ORD-2024-00808-1,24 augusti 2024,2024-08-20,SKU-KB001,Mekaniskt Tangentbord K7,Tillbehör,1,1299.00,Stockholm,Privat,Kort,KND-47871,Levererad,NaN,NaN,NaN
2330,ORD-2024-01394,ORD-2024-01394-2,2024-03-14,2024-03-18,SKU-MS002,Ergonomisk Mus Pro,Tillbehör,1st,699.00,Uppsala,Företag,Faktura,KND-98270,Levererad,NaN,NaN,NaN
538,ORD-2024-00334,ORD-2024-00334-1,2024-05-18,2024/05/21,SKU-MN001,"Bildskärm 27"" UHD",Bildskärmar,1,4999.00,Göteborg,Privat,FAKTURA,KND-95748,Levererad,Varken bra eller dåligt. Fungerar.,2024-06-03,3.0
2684,ORD-2024-01606,ORD-2024-01606-1,2024-10-23,2024-10-26,SKU-LP001,Laptop Pro 15,Datorer,1,14999.00,Göteborg,Företag,Faktura,KND-31939,Levererad,NaN,NaN,NaN
2685,ORD-2024-01606,ORD-2024-01606-2,2024-10-23,2024-10-26,SKU-LP001,Laptop Pro 15,Datorer,2,14999.00,Göteborg,b2b,Faktura,KND-31939,Levererad,Kan varmt rekommendera denna produkt.,2024-11-01,4.0
23,ORD-2024-00016,ORD-2024-00016-1,2024-04-27,2024-05-01,SKU-SD001,Extern SSD 1TB,Lagring,1,1199.00,Västerås,Privat,Mobilbetalning,KND-52160,Levererad,NaN,NaN,NaN
944,ORD-2024-00583,ORD-2024-00583-1,2024-03-06,2024-03-08,SKU-KB001,Mekaniskt Tangentbord K7,Tillbehör,2,1299.00,Stockholm,Privat,Swish,KND-36658,Levererad,"Leverans på två dagar, imponerad!",2024-03-13,5.0


In [182]:
df_raw["region"].value_counts()

region
Stockholm     834
Göteborg      434
Malmö         236
Uppsala       192
Norrland      125
Örebro        117
Linköping     117
Västerås       75
STOCKHOLM      50
Sthml          44
stockholm      42
STHLM          39
uppsala        25
Sthlm          25
göteborg       24
GÖTEBORG       22
UPPSALA        21
Gothenburg     19
MALMÖ          16
Gbg            14
malmo          12
GBGB           11
LINKÖPING      11
Orebro         10
Vasteras       10
örebro          9
ÖREBRO          9
norrland        9
linköping       8
NORRLAND        8
Linkoping       8
Malmo           8
västerås        7
Norr            7
VÄSTERÅS        7
malmö           7
Name: count, dtype: int64

In [183]:
df_raw["kundtyp"].value_counts()

kundtyp
Privat       1535
Företag       783
privat         66
b2c            64
Konsument      60
PRIVAT         58
B2C            45
B2B            39
b2b            35
FÖRETAG        29
Firma          27
företag        26
Name: count, dtype: int64

In [184]:
df_raw["betalmetod"].value_counts()

betalmetod
Faktura           938
Kort              775
Swish             550
Invoice            51
FAKTURA            50
faktura            44
Kreditkort         36
KORT               33
swish              31
SWISH              31
kort               30
Mobilbetalning     29
Visa               28
Mastercard         25
Name: count, dtype: int64

In [185]:
df_raw["leveransstatus"].value_counts()

leveransstatus
Levererad          2005
Under transport     152
Retur               132
Skickad              95
Mottagen             92
levererad            86
LEVERERAD            68
Returnerad           10
Återsänd              8
På väg                7
retur                 6
under transport       6
RETUR                 4
UNDER TRANSPORT       2
Name: count, dtype: int64

In [186]:
df_raw["orderrad_id"].isna().sum()


np.int64(0)

In [187]:
df_raw.duplicated().sum()

np.int64(67)

In [188]:
df_raw["orderrad_id"].duplicated().sum()


np.int64(67)

In [189]:
# STEG 1: Ersätt svenska månader FÖRST
swedish_to_english = {
    'januari': 'January', 'februari': 'February', 'mars': 'March',
    'april': 'April', 'maj': 'May', 'juni': 'June',
    'juli': 'July', 'augusti': 'August', 'september': 'September',
    'oktober': 'October', 'november': 'November', 'december': 'December'
}
 
for col in ['orderdatum', 'leveransdatum', 'recensionsdatum']:
    df_raw[col] = df_raw[col].astype(str)
    for swe, eng in swedish_to_english.items():
        df_raw[col] = df_raw[col].str.replace(swe, eng, case=False)
 
# STEG 2: Sedan parsa
for col in ['orderdatum', 'leveransdatum', 'recensionsdatum']:
    df_raw[col] = pd.to_datetime(df_raw[col], format='mixed', dayfirst=True, errors='coerce')

In [190]:
# Kolla min/max 
for col in ['orderdatum', 'leveransdatum', 'recensionsdatum']:     
    print(f"{col}: {df_raw[col].min().date()} → {df_raw[col].max().date()}")

orderdatum: 2024-01-01 → 2024-12-31
leveransdatum: 2023-12-27 → 2025-01-07
recensionsdatum: 2024-01-05 → 2025-01-14


In [191]:
print(pd.__version__)

2.3.3


In [192]:
# Kör detta och visa output
test = pd.to_datetime(['01/07/2025'], format='mixed', dayfirst=True)
print(test)  # Ska bli 2025-07-01 om dayfirst=False, 2025-01-07 om dayfirst=True

DatetimeIndex(['2025-07-01'], dtype='datetime64[ns]', freq=None)


In [193]:
df_test = pd.read_csv('nordtech_data.csv')  # Ändra till din sökväg
 
# Visa max RÅVÄRDEN (före parsning)
print("Max råvärden (strängar):")
print(f"leveransdatum: {sorted(df_test['leveransdatum'].dropna().unique())[-5:]}")
print(f"recensionsdatum: {sorted(df_test['recensionsdatum'].dropna().astype(str).unique())[-5:]}")


Max råvärden (strängar):
leveransdatum: ['September 08, 2024', 'September 09, 2024', 'September 26, 2024', 'September 28, 2024', 'September 29, 2024']
recensionsdatum: ['November 07, 2024', 'November 26, 2024', 'October 08, 2024', 'October 10, 2024', 'October 18, 2024']


In [223]:
mask_fel_datum = df_raw["leveransdatum"] < df_raw["orderdatum"]

antal_fel = mask_fel_datum.sum()
andel_fel = antal_fel / len(df_raw)

antal_fel, andel_fel


(np.int64(59), np.float64(0.021322732200939647))

In [194]:
df_raw["order_id"].duplicated().sum()

np.int64(1110)

In [195]:
dupes = df_raw[df_raw["orderrad_id"].duplicated(keep=False)].sort_values("orderrad_id")
dupes.head(10)


,order_id,orderrad_id,orderdatum,leveransdatum,produkt_sku,produktnamn,kategori,antal,pris_per_enhet,region,kundtyp,betalmetod,kund_id,leveransstatus,recension_text,recensionsdatum,betyg
613,ORD-2024-00070,ORD-2024-00070-1,2024-11-18,2024-11-20,SKU-KB002,Kompakt Tangentbord Mini,Tillbehör,1,599.00,Stockholm,Privat,Kort,KND-13904,Levererad,Riktigt nöjd! Använder den dagligen.,2024-11-23,4.0
107,ORD-2024-00070,ORD-2024-00070-1,2024-11-18,2024-11-20,SKU-KB002,Kompakt Tangentbord Mini,Tillbehör,1,599.00,Stockholm,Privat,Kort,KND-13904,Levererad,Riktigt nöjd! Använder den dagligen.,2024-11-23,4.0
181,ORD-2024-00110,ORD-2024-00110-1,2024-07-11,2024-07-14,SKU-MS001,Trådlös Mus X1,Tillbehör,2,399 kr,Gbg,privat,Kort,KND-13544,Levererad,"Leverans på två dagar, imponerad!",2024-07-21,4.0
1403,ORD-2024-00110,ORD-2024-00110-1,2024-07-11,2024-07-14,SKU-MS001,Trådlös Mus X1,Tillbehör,2,399 kr,Gbg,privat,Kort,KND-13544,Levererad,"Leverans på två dagar, imponerad!",2024-07-21,4.0
2003,ORD-2024-00111,ORD-2024-00111-1,2024-06-21,2024-06-26,SKU-MN002,"Bildskärm 24"" FHD",Bildskärmar,1,2499.00,NORRLAND,Företag,Faktura,KND-39076,Levererad,Leveransskada - kartongen var helt demolerad.,2024-07-06,2.0
182,ORD-2024-00111,ORD-2024-00111-1,2024-06-21,2024-06-26,SKU-MN002,"Bildskärm 24"" FHD",Bildskärmar,1,2499.00,NORRLAND,Företag,Faktura,KND-39076,Levererad,Leveransskada - kartongen var helt demolerad.,2024-07-06,2.0
224,ORD-2024-00138,ORD-2024-00138-2,2024-09-17,2024-09-21,SKU-HS002,Headset Budget,Ljud,2,499.00,Linköping,b2b,Kort,KND-41713,Levererad,Fungerar inte som det ska. Måste returnera.,2024-09-29,1.0
1682,ORD-2024-00138,ORD-2024-00138-2,2024-09-17,2024-09-21,SKU-HS002,Headset Budget,Ljud,2,499.00,Linköping,b2b,Kort,KND-41713,Levererad,Fungerar inte som det ska. Måste returnera.,2024-09-29,1.0
276,ORD-2024-00168,ORD-2024-00168-2,2024-07-31,2024-08-02,SKU-MS002,Ergonomisk Mus Pro,Tillbehör,1,699.00,Göteborg,privat,Faktura,KND-30388,Levererad,Prisvärt och bra kvalitet. Kommer köpa igen.,2024-08-14,5.0
383,ORD-2024-00168,ORD-2024-00168-2,2024-07-31,2024-08-02,SKU-MS002,Ergonomisk Mus Pro,Tillbehör,1,699.00,Göteborg,privat,Faktura,KND-30388,Levererad,Prisvärt och bra kvalitet. Kommer köpa igen.,2024-08-14,5.0


In [196]:
(df_raw[["order_id", "orderrad_id"]]
 .duplicated()
 .sum())


np.int64(67)

In [197]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2767 entries, 0 to 2766
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   order_id         2767 non-null   object        
 1   orderrad_id      2767 non-null   object        
 2   orderdatum       2767 non-null   datetime64[ns]
 3   leveransdatum    2767 non-null   datetime64[ns]
 4   produkt_sku      2767 non-null   object        
 5   produktnamn      2767 non-null   object        
 6   kategori         2767 non-null   object        
 7   antal            2767 non-null   object        
 8   pris_per_enhet   2767 non-null   object        
 9   region           2612 non-null   object        
 10  kundtyp          2767 non-null   object        
 11  betalmetod       2651 non-null   object        
 12  kund_id          2767 non-null   object        
 13  leveransstatus   2673 non-null   object        
 14  recension_text   1355 non-null   object 

Frågeställningar : Se vilka produkter som säljs mest samt vilka kategorier. Se vilken kategori som genererar mest intäkter

# EDA

1. Datasetets grain är på orderrad-nivå, men det finns dubletter som behöver hanteras.

2. Orderdatum är en sträng som behöver göras om till datetime, samma gäller för leveransdatum. 

3. Antal ska ändras om till en int. 

4. Pris_per_enhet ska göras om till float, samt standardisera värdet i pris_per_enhet.

5. Region beöver standardiseras och nullvärden hanteras. 

6. Kundtyp behöver standardiseras till rätt namn. 

7. betalmetod behöver standardiseras till rätt namn, samt hantera nullvärden.

8. Leveransstatus behöver standardiseras till rätt namn, samt hantera nullvärden.

9. Behöver göra om recensionsdatum till datetime. 


In [198]:
#1. fixa dubletter i orderrader
def clean_orderrader(df):
    df_clean = df.copy()
    before = len(df_clean)

    # 1) Ta bort exakt identiska rader (konservativt, riskerar inte databortfall)
    df_clean = df_clean.drop_duplicates().reset_index(drop=True)

    removed = before - len(df_clean)
    print(f"Tagit bort dubletter (identiska rader): {removed}")

    # 2) Kontroll: finns det fortfarande dubletter på orderrad_id?
    # (dvs samma orderrad_id men raderna skiljer sig i någon kolumn)
    if "orderrad_id" in df_clean.columns:
        remaining_dup_ids = df_clean["orderrad_id"].duplicated(keep=False).sum()
        if remaining_dup_ids > 0:
            print(f"VARNING: {remaining_dup_ids} rader delar samma orderrad_id efter drop_duplicates().")
            print("Detta tyder på logiska dubletter (samma nyckel men olika innehåll).")

    return df_clean




In [199]:
#2. fixa orderdatum
def clean_orderdatum(df):
    df_clean = df.copy()

    # 1) Normalisera text (svenska månadsnamn -> engelska) + ta bort kommatecken
    month_map = {
        "januari": "january",
        "februari": "february",
        "mars": "march",
        "april": "april",
        "maj": "may",
        "juni": "june",
        "juli": "july",
        "augusti": "august",
        "september": "september",
        "oktober": "october",
        "november": "november",
        "december": "december",
    }

    s = df_clean["orderdatum"].astype("string").str.strip().str.lower()
    s = s.str.replace(",", "", regex=False)

    # Byt ut svenska månadsnamn oavsett var de ligger i strängen
    for sv, en in month_map.items():
        s = s.str.replace(rf"\b{sv}\b", en, regex=True)

    # 2) Parse till datetime
    df_clean["orderdatum"] = pd.to_datetime(
        s,
        errors="coerce",
        dayfirst=True,
        format="mixed",
    )

    invalid_dates = df_clean["orderdatum"].isna().sum()
    if invalid_dates > 0:
        print(f"Hittade {invalid_dates} ogiltiga datum i 'orderdatum', sätter dessa till NaT.")
    return df_clean





Här gjorde jag om orderdatum till datetime och eftersom att det fanns olika format och även bokstäver i samt på både svenska och engelska så fick jag standardisera språket sedan lägga in regex för att den skulle kunna konvertera om vart än månaden var skriven i texten. 

In [200]:
#2.1 fixa leveransdatum
def clean_leveransdatum(df):
    df_clean = df.copy()

    # 1) Normalisera text (svenska månadsnamn -> engelska) + ta bort kommatecken
    month_map = {
        "januari": "january",
        "februari": "february",
        "mars": "march",
        "april": "april",
        "maj": "may",
        "juni": "june",
        "juli": "july",
        "augusti": "august",
        "september": "september",
        "oktober": "october",
        "november": "november",
        "december": "december",
    }

    s = df_clean["leveransdatum"].astype("string").str.strip().str.lower()
    s = s.str.replace(",", "", regex=False)

    # Byt ut svenska månadsnamn oavsett var de ligger i strängen
    for sv, en in month_map.items():
        s = s.str.replace(rf"\b{sv}\b", en, regex=True)

    # 2) Parse till datetime
    df_clean["leveransdatum"] = pd.to_datetime(
        s,
        errors="coerce",
        dayfirst=True,
        format="mixed",
    )

    invalid_dates = df_clean["leveransdatum"].isna().sum()
    if invalid_dates > 0:
        print(f"Hittade {invalid_dates} ogiltiga datum i 'leveransdatum', sätter dessa till NaT.")

    return df_clean





Här gjorde jag om orderdatum till datetime och eftersom att det fanns olika format och även bokstäver i samt på både svenska och engelska så fick jag standardisera språket sedan lägga in regex för att den skulle kunna konvertera om vart än månaden var skriven i texten. 

In [201]:
#3.
def clean_antal(df):
    df_clean = df.copy()
    
    # Konvertera 'antal' till numeriskt format
    df_clean['antal'] = pd.to_numeric(
        df_clean['antal'], 
        errors='coerce')

    # Hantera negativa och nollvärden
    invalid_antal = (df_clean['antal'] <= 0).sum()
    if invalid_antal > 0:
        print(f"Hittade {invalid_antal} ogiltiga värden i 'antal' (negativa eller noll), sätter dessa till NaN.")
        
        df_clean.loc[df_clean['antal'] <= 0, 'antal'] = np.nan

    df_clean["antal"] = df_clean["antal"].astype("Int64")

    return df_clean



Här gjorde jag om antal från en sträng till int64.

In [202]:
#4.
def clean_pris_per_enhet(df):
    df_clean = df.copy()

    df_clean['pris_per_enhet'] = (
        df_clean['pris_per_enhet']
        .astype(str)
        .str.replace(",", ".", regex=False)              # hantera decimal-komma
        .str.replace(r"[^0-9.]", "", regex=True)
    )
    
    # Konvertera 'pris_per_enhet' till numeriskt format
    df_clean['pris_per_enhet'] = pd.to_numeric(
        df_clean['pris_per_enhet'],
        errors='coerce')

    # Hantera negativa och nollvärden
    invalid_pris = (df_clean['pris_per_enhet'] <= 0).sum()
    if invalid_pris > 0:
        print(f"Hittade {invalid_pris} ogiltiga värden i 'pris_per_enhet' (negativa eller noll), sätter dessa till NaN.")
        
        df_clean.loc[df_clean['pris_per_enhet'] <= 0, 'pris_per_enhet'] = np.nan

    return df_clean




I pris_per_enhet fanns en del fel så jag fick först standardisera hur jag ville att det skulle se ut. så jag valde att ta bort att bokstäver och ",". Sedan göra om strängen till float för att enklare kunna göra beräkningar senare. 

In [203]:
#5.
def standardisera_region(df):
    df_clean = df.copy()
    df_clean["region_raw"] = df_clean["region"]

    def klassificera_region(val):
        if pd.isna(val):
            return "Okänd"

        v = str(val).lower().strip()

        if v in ["stockholm", "sthlm", "sthl", "sthml"]:
            return "Stockholm"

        if v in ["uppsala"]:
            return "Uppsala"

        if v in ["göteborg", "gothenburg", "gbg", "gbgb"]:
            return "Göteborg"

        if v in ["malmö", "malmo"]:
            return "Malmö"

        if v in ["norrland", "norr"]:
            return "Norrland"

        if v in ["örebro", "orebro"]:
            return "Örebro"

        if v in ["västerås", "vasteras"]:
            return "Västerås"

        if v in ["linköping", "linkoping"]:
            return "Linköping"

        return "Okänd"

    df_clean["region"] = df_clean["region"].apply(klassificera_region)
    return df_clean




I region tabellen så hade vi mycket olika stavningar och förkortningar. Så här gjorde jag först om det till lower och tog bort whitspace och sedan kunde jag skriva "om detta finns i detta gör det till detta. om inte gör det till "okänd". Mest för att slippa hårkoda in alla typer av olika stora och små bokstäver som skrivs. 

In [204]:
#6.
def standardisera_kundtyp(df):
    df_clean = df.copy()
    df_clean["kundtyp_raw"] = df_clean["kundtyp"]

    def klassificera_kundtyp(val):
        if pd.isna(val):
            return "Okänd"

        v = str(val).lower().strip()

        # Privat / B2C
        if any(x in v for x in ["privat", "b2c", "konsument"]):
            return "Privat"

        # Företag / B2B
        if any(x in v for x in ["företag", "firma", "b2b"]):
            return "Företag"

        return "Okänd"

    df_clean["kundtyp"] = df_clean["kundtyp"].apply(klassificera_kundtyp)
    return df_clean




Här gjorde jag samma som i den tidigare funktionen. 

In [205]:
#7. 
def standardisera_betalmetod(df):
    df_clean = df.copy()
    df_clean["betalmetod_raw"] = df_clean["betalmetod"]

    def klassificera_betalmetod(val):
        if pd.isna(val):
            return "Okänd"

        v = str(val).lower().strip()

        # Faktura
        if any(x in v for x in ["faktura", "invoice"]):
            return "Faktura"

        # Swish
        if "swish" in v:
            return "Swish"

        # Kortbetalning
        if any(x in v for x in ["kort", "visa", "mastercard", "kredit"]):
            return "Kort"

        # Mobilbetalning
        if "mobil" in v:
            return "Mobilbetalning"

        return "Okänd"

    df_clean["betalmetod"] = df_clean["betalmetod"].apply(klassificera_betalmetod)
    return df_clean





Jag gjorde samma som i de 2 tidigare funktionerna men i denna så såg jag även att mobiltelefon utger en väldigt liten del vilket gör att när jag gör beräkningar så kommer jag inte att ta med den för den kommer göra beräkningen dålig. 

In [206]:
#8.
def standardisera_leveransstatus(df):
    df_clean = df.copy()

    # Grundläggande textnormalisering
    status = (
        df_clean['leveransstatus']
        .astype(str)
        .str.strip()
        .str.lower()
    )

    df_clean['leveransstatus_std'] = np.select(
        [
            status.str.contains(r"levererad|mottagen"),
            status.str.contains(r"transport|skickad|på väg"),
            status.str.contains(r"retur|return|återsänd"),
        ],
        [
            "Levererad",
            "Under transport",
            "Retur",
        ],
        default="Okänd"
    )

    return df_clean




Denna funktionen så standardiserade jag leveransstatus. Även i detta fall valde jag att inte använda en hårdkodad mapping. Men tog hjälp av rexex att begränsa till 4 kategorier och klassificeringen gjordes vektoriserat med hjälp av np.select.

In [207]:
#9.
def clean_recensionsdatum(df):
    df_clean = df.copy()

    # 1) Normalisera text (svenska månadsnamn -> engelska) + ta bort kommatecken
    month_map = {
        "januari": "january",
        "februari": "february",
        "mars": "march",
        "april": "april",
        "maj": "may",
        "juni": "june",
        "juli": "july",
        "augusti": "august",
        "september": "september",
        "oktober": "october",
        "november": "november",
        "december": "december",
    }

    s = df_clean["recensionsdatum"].astype("string").str.strip().str.lower()
    s = s.str.replace(",", "", regex=False)

    # Byt ut svenska månadsnamn oavsett var de ligger i strängen
    for sv, en in month_map.items():
        s = s.str.replace(rf"\b{sv}\b", en, regex=True)

    # 2) Parse till datetime
    df_clean["recensionsdatum"] = pd.to_datetime(
        s,
        errors="coerce",
        dayfirst=True,
        format="mixed",
    )

    invalid_dates = df_clean["recensionsdatum"].isna().sum()
    if invalid_dates > 0:
        print(f"Hittade {invalid_dates} ogiltiga datum i 'recensionsdatum', sätter dessa till NaT.")
    return df_clean



Även i recensionsdatum fanns samma fel i hur datumen var skrivna så använde mig av samma funktion men bytte ut kolumnnamnet. 1412 ogiltiga datum finns för att det var så många saknade rader i den kolumnen. 

# Skapa data-dictionary

In [208]:

# Dokumentation baserad på min EDA
COLUMN_DOCS = {
    "order_id": {
        "description": "Unikt ID för varje order.",
        "allowed_values_or_format": "Integer",
        "rules_and_notes": "Kan förekomma flera gånger eftersom datasetet är på orderrad-nivå."
    },
    "orderrad_id": {
        "description": "Unikt ID för varje orderrad.",
        "allowed_values_or_format": "Integer",
        "rules_and_notes": "Identifierar datasetets grain (orderrad-nivå). Dubletter identifierades och hanterades."
    },
    "orderdatum": {
        "description": "Datum då ordern skapades.",
        "allowed_values_or_format": "datetime",
        "rules_and_notes": "Konverterad från sträng till datetime. Blandade datumformat (svenska/engelska månadsnamn) standardiserades."
    },
    "leveransdatum": {
        "description": "Datum då ordern levererades.",
        "allowed_values_or_format": "datetime",
        "rules_and_notes": "Konverterad från sträng till datetime. Ogiltiga datum sattes till NaT."
    },
    "antal": {
        "description": "Antal produkter på orderraden.",
        "allowed_values_or_format": "Integer ≥ 1",
        "rules_and_notes": "Konverterad till int för korrekt beräkning av intäkter."
    },
    "pris_per_enhet": {
        "description": "Pris per enhet för produkten på orderraden.",
        "allowed_values_or_format": "Float, t.ex. 1200.00",
        "rules_and_notes": "Rensad från bokstäver, valutor och symboler samt standardiserad till numeriskt format."
    },
    "region": {
        "description": "Geografisk region kopplad till ordern.",
        "allowed_values_or_format": "Kategorisk text",
        "rules_and_notes": "Standardiserad till konsekventa regionnamn. Saknade värden hanterades."
    },
    "kundtyp": {
        "description": "Typ av kund (t.ex. privat eller företag).",
        "allowed_values_or_format": "Kategorisk text",
        "rules_and_notes": "Standardiserad till konsekventa benämningar."
    },
    "betalmetod": {
        "description": "Betalningsmetod som användes vid köpet.",
        "allowed_values_or_format": "Kategorisk text",
        "rules_and_notes": "Standardiserad till konsekventa namn. Nullvärden hanterades."
    },
    "leveransstatus": {
        "description": "Leveransstatus för ordern.",
        "allowed_values_or_format": "Kategorisk text",
        "rules_and_notes": "Standardiserad via regelbaserad textklassificering (t.ex. Levererad, Under transport, Retur)."
    },
    "recensionsdatum": {
        "description": "Datum då recensionen lämnades.",
        "allowed_values_or_format": "datetime",
        "rules_and_notes": "Konverterad från sträng till datetime. Ogiltiga värden sattes till NaT."
    }
}

def skapa_data_dictionary(df_clean: pd.DataFrame, docs: dict) -> pd.DataFrame:
    rows = []
    n = len(df_clean)

    for col in df_clean.columns:
        s = df_clean[col]
        rows.append({
            "column": col,
            "dtype": str(s.dtype),
            "non_null_pct": round(s.notna().mean() * 100, 1),
            "n_unique": s.nunique(dropna=True),
            "example_value": s.dropna().iloc[0] if s.dropna().shape[0] > 0 else np.nan,
            "description": docs.get(col, {}).get("description", ""),
            "allowed_values_or_format": docs.get(col, {}).get("allowed_values_or_format", ""),
            "rules_and_notes": docs.get(col, {}).get("rules_and_notes", ""),
        })

    return pd.DataFrame(rows).sort_values("column").reset_index(drop=True)









# Läsa in valideringsdatasetet

In [209]:
df_val_raw = pd.read_csv("nordtech_validation.csv")

# Feature Engineering 

In [210]:
def addera_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["rad_intakt"] = df["antal"] * df["pris_per_enhet"]

    weekday_map = {
        "Monday": "Måndag", "Tuesday": "Tisdag", "Wednesday": "Onsdag",
        "Thursday": "Torsdag", "Friday": "Fredag",
        "Saturday": "Lördag", "Sunday": "Söndag"
    }
    df["veckodag"] = df["orderdatum"].dt.day_name().map(weekday_map)

    df["månad_num"] = df["orderdatum"].dt.month
    month_map = {
        1:"Januari",2:"Februari",3:"Mars",4:"April",5:"Maj",6:"Juni",
        7:"Juli",8:"Augusti",9:"September",10:"Oktober",11:"November",12:"December"
    }
    df["månad"] = df["månad_num"].map(month_map)

    iso = df["orderdatum"].dt.isocalendar()
    df["iso_år"] = iso["year"].astype(int)
    df["iso_vecka"] = iso["week"].astype(int)
    df["år_vecka"] = df["iso_år"].astype(str) + "-W" + df["iso_vecka"].astype(str).str.zfill(2)

    return df


# Sentimentanalys

In [211]:
SYSTEM_PROMPT = """
Du är en analytisk sentimentklassificerare.

Din uppgift är att klassificera kundrecensioner från en e-handelsplattform
i exakt EN av följande tre kategorier:

- positiv
- neutral
- negativ

Regler:
- Svara endast med ett av orden: positiv, neutral eller negativ.
- Lägg inte till förklaringar eller extra text.
- Om recensionen är tom eller oklar, klassificera som neutral.
- Recensionerna är skrivna på svenska.
"""


In [212]:
def addera_sentiment(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    def classify(text):
        if pd.isna(text) or text.strip() == "":
            return "neutral"

        response = llm.invoke([
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": text[:500]}
        ])

        return response.content.strip().lower()

    df["sentiment"] = df["recension_text"].apply(classify)
    return df


In [213]:
# Visar att LLM fungerar och kan klassificera sentiment

df_test = df_raw.loc[
    df_raw["recension_text"].notna(),
    ["recension_text"]
].sample(5, random_state=42)

df_test["sentiment_test"] = df_test["recension_text"].apply(
    lambda x: llm.invoke([
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": x[:500]}
    ]).content.strip().lower()
)

df_test


,recension_text,sentiment_test
105,"Förväntade mig mer för priset, men den duger.",neutral
2229,"Förväntade mig mer för priset, men den duger.",neutral
1020,"Leverans på två dagar, imponerad!",positiv
2259,Helt underbar kvalitet för pengarna.,positiv
1891,Överträffade mina förväntningar. 5 av 5!,positiv


# Fullständig Transform för validering

In [ ]:
def run_pipeline(df_raw: pd.DataFrame) -> pd.DataFrame:
    df_clean = df_raw.copy()

    # 1) Grain & dubbletter
    df_clean = clean_orderrader(df_clean)

    # 2) Datumstandardisering
    df_clean = clean_orderdatum(df_clean)
    df_clean = clean_leveransdatum(df_clean)
    df_clean = clean_recensionsdatum(df_clean)
    

    # 3) Numeriska värden
    df_clean = clean_pris_per_enhet(df_clean)
    df_clean = clean_antal(df_clean)

    # 4) Kategoriska standardiseringar
    df_clean = standardisera_region(df_clean)
    df_clean = standardisera_kundtyp(df_clean)
    df_clean = standardisera_betalmetod(df_clean)
    df_clean = standardisera_leveransstatus(df_clean)

    # 5) Feature engineering
    df_clean = addera_features(df_clean)

    # 6) Sentimentanalys
    #df_clean = addera_sentiment(df_clean)

    return df_clean

# --- TRAIN ---
df_clean = run_pipeline(df_raw)




print("✓ Pipeline körd och validerad på TRAIN")


# --- VALIDATION ---
df_val_clean = run_pipeline(df_val_raw)




print("✓ Pipeline körd och validerad på VALIDATION")


Tagit bort dubletter (identiska rader): 67
Hittade 1383 ogiltiga datum i 'recensionsdatum', sätter dessa till NaT.
✓ Pipeline körd och validerad på TRAIN
Tagit bort dubletter (identiska rader): 11
Hittade 229 ogiltiga datum i 'recensionsdatum', sätter dessa till NaT.
✓ Pipeline körd och validerad på VALIDATION


# Importera till SQLite-Databas

In [215]:
import sqlite3

conn = sqlite3.connect("ehandel_data_clean.db")

df_clean.to_sql(
    "orderdata", 
    conn, 
    if_exists="replace", 
    index=False
)

print("✓ TRAIN-data laddad till SQLite (replace)")


✓ TRAIN-data laddad till SQLite (replace)


In [216]:
pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)


,name
0,orderdata


Kollar så att df_clean har laddats in på rätt sätt.

In [217]:
rows_df = len(df_clean)

rows_sql = pd.read_sql(
    "SELECT COUNT(*) AS n FROM orderdata",
    conn
)["n"].iloc[0]

print("Rader i df_clean:", rows_df)
print("Rader i SQLite:", rows_sql)

assert rows_df == rows_sql, "Antal rader matchar inte!"
print("✓ Antal rader matchar")


Rader i df_clean: 2700
Rader i SQLite: 2700
✓ Antal rader matchar


Dubbelkollar så att det inte har tappats eller dubblats några rader vid load.

In [218]:
df_val_clean.to_sql(
    "orderdata",
    conn,
    if_exists="append",
    index=False
)

print("✓ VALIDATION-data laddad till SQLite (append)")


✓ VALIDATION-data laddad till SQLite (append)


In [219]:
train_rows = len(df_clean)
val_rows = len(df_val_clean)

sql_rows = pd.read_sql(
    "SELECT COUNT(*) AS n FROM orderdata",
    conn
)["n"].iloc[0]

print("Train rader:", train_rows)
print("Validation rader:", val_rows)
print("SQLite rader:", sql_rows)

assert sql_rows == train_rows + val_rows, "Antal rader i SQLite matchar inte train + validation!"
print("✓ Antal rader matchar (train + validation)")


Train rader: 2700
Validation rader: 450
SQLite rader: 3150
✓ Antal rader matchar (train + validation)


Dubbelkollar så att antal rader matchar efter load.

In [220]:
cols_df = set(df_clean.columns)
cols_sql = set(pd.read_sql("PRAGMA table_info(orderdata);", conn)["name"])

missing = cols_df - cols_sql
extra = cols_sql - cols_df

print("Saknas i SQLite:", missing)
print("Extra i SQLite:", extra)

assert not missing and not extra, "Kolumnschema matchar inte!"
print("✓ Kolumner matchar")


Saknas i SQLite: set()
Extra i SQLite: set()
✓ Kolumner matchar


Dubbelkollar så att det inte saknas eller är några exra rader efter load.

In [221]:
pd.read_sql("SELECT * FROM orderdata LIMIT 10;", conn)


,order_id,orderrad_id,orderdatum,leveransdatum,produkt_sku,produktnamn,kategori,antal,pris_per_enhet,region,...,kundtyp_raw,betalmetod_raw,leveransstatus_std,rad_intakt,veckodag,månad_num,månad,iso_år,iso_vecka,år_vecka
0,ORD-2024-00001,ORD-2024-00001-1,2024-05-19 00:00:00,2024-05-22 00:00:00,SKU-WC001,Webbkamera HD,Tillbehör,1.0,799.0,Uppsala,...,Privat,Kort,Levererad,799.0,Söndag,5,Maj,2024,20,2024-W20
1,ORD-2024-00002,ORD-2024-00002-1,2024-12-02 00:00:00,2024-12-05 00:00:00,SKU-HB001,USB-C Hub 7-port,Tillbehör,1.0,549.0,Göteborg,...,Privat,Swish,Levererad,549.0,Måndag,12,December,2024,49,2024-W49
2,ORD-2024-00003,ORD-2024-00003-1,2024-12-31 00:00:00,2025-01-03 00:00:00,SKU-SD001,Extern SSD 1TB,Lagring,1.0,1199.0,Okänd,...,Företag,Faktura,Levererad,1199.0,Tisdag,12,December,2025,1,2025-W01
3,ORD-2024-00003,ORD-2024-00003-2,2024-12-31 00:00:00,2025-01-03 00:00:00,SKU-SD002,Extern SSD 500GB,Lagring,10.0,699.0,Stockholm,...,Företag,FAKTURA,Levererad,6990.0,Tisdag,12,December,2025,1,2025-W01
4,ORD-2024-00003,ORD-2024-00003-3,2024-12-31 00:00:00,2025-01-03 00:00:00,SKU-MS001,Trådlös Mus X1,Tillbehör,1.0,399.0,Stockholm,...,Företag,Faktura,Okänd,399.0,Tisdag,12,December,2025,1,2025-W01
5,ORD-2024-00004,ORD-2024-00004-1,2024-04-22 00:00:00,2024-04-26 00:00:00,SKU-WC001,Webbkamera HD,Tillbehör,NaN,799.0,Örebro,...,Privat,Faktura,Levererad,NaN,Måndag,4,April,2024,17,2024-W17
6,ORD-2024-00005,ORD-2024-00005-1,2024-07-01 00:00:00,2024-07-05 00:00:00,SKU-HB001,USB-C Hub 7-port,Tillbehör,10.0,549.0,Örebro,...,FÖRETAG,Faktura,Levererad,5490.0,Måndag,7,Juli,2024,27,2024-W27
7,ORD-2024-00005,ORD-2024-00005-2,2024-07-01 00:00:00,2024-07-05 00:00:00,SKU-HS001,Headset Pro ANC,Ljud,3.0,1899.0,Örebro,...,Företag,FAKTURA,Levererad,5697.0,Måndag,7,Juli,2024,27,2024-W27
8,ORD-2024-00005,ORD-2024-00005-3,2024-07-01 00:00:00,2024-07-05 00:00:00,SKU-KB002,Kompakt Tangentbord Mini,Tillbehör,2.0,599.0,Örebro,...,Företag,Faktura,Levererad,1198.0,Måndag,7,Juli,2024,27,2024-W27
9,ORD-2024-00005,ORD-2024-00005-4,2024-07-01 00:00:00,2024-07-05 00:00:00,SKU-MS001,Trådlös Mus X1,Tillbehör,3.0,399.0,Örebro,...,Företag,Faktura,Levererad,1197.0,Måndag,7,Juli,2024,27,2024-W27


"sanity check" för att se så att det inte har blivit strul med datan när den skickades över till databasen.